In [1]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import config as cfg
import os

import torch
from torch import nn
from torch.utils.data import DataLoader

from Data.data_loading import load_and_preprocess_data, create_tensor_from_dataframe, create_sequences, create_dataloaders 
from Training.train_matt import Trainer
from Training.basicEval import plotLoss, plotAccuracy, reportFinalMetrics, reportMultiFinalMetrics, plotMultiAccuracy, plotMultiLoss
from Model.model_split import FrameTransformer, print_model_info

from Training.customLoss import ADELoss, FDELoss, RMSELoss

Using GPU


In [ ]:
root_dir = os.getcwd()  # Use current working directory as root
data_dir = os.path.join(root_dir, 'Data')
csv_dir = os.path.join(data_dir, 'one_csv')
csv_file = os.path.join(csv_dir, 'michael_10s_processed.csv')
model_dir = os.path.join(root_dir, 'Model', 'Saved_Model')
model_path = os.path.join(model_dir, 'mse_model.pth')

print("Data directory: ", data_dir)
print("CSV directory: ", csv_dir)
print("CSV file: ", csv_file)


model_dir = os.path.join(root_dir, 'Model')
save_model_dir = os.path.join(model_dir, 'Saved_Model')
print("Model directory: ", model_dir)
print("Saved model directory: ", save_model_dir)


Data directory:  /home/jaskin/Deep-Learning-Project/Data
CSV directory:  /home/jaskin/Deep-Learning-Project/Data/one_csv
CSV file:  /home/jaskin/Deep-Learning-Project/Data/one_csv/michael_10s_processed.csv
Model directory:  /home/jaskin/Deep-Learning-Project/Model
Saved model directory:  /home/jaskin/Deep-Learning-Project/Model/Saved_Model


In [3]:
import numpy as np

def test_model(model, test_loader, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()  # Set the model to evaluation mode
    
    all_predictions = []
    all_targets = []
    total_loss = 0.0
    loss_fn = torch.nn.MSELoss()

    with torch.no_grad():  # Disable gradient computation for faster testing
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            outputs = model(inputs)  # Make predictions on the batch
            
            batch_loss = loss_fn(outputs, targets)
            total_loss += batch_loss.item() * inputs.size(0) # Weighted by batch size
            
            all_predictions.append(outputs.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # Concatenate all predictions and targets from batches
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    
    # Calculate overall metrics
    avg_loss = total_loss / len(test_loader.dataset)
    print(f'Test Loss (MSE): {avg_loss:.6f}')

    # Reshape for metric calculation if necessary (assuming [batch, pred_len, num_ids, 2])
    # Adjust axis based on your actual output shape and how ADE/RMSE should be computed
    if len(all_predictions.shape) == 4:
        # Example: Calculate error per point across batch, pred_len, num_ids
        errors = np.linalg.norm(all_predictions - all_targets, axis=-1) # Norm along the last axis (X, Y)
        ade = np.mean(errors)
        rmse = np.sqrt(np.mean((all_predictions - all_targets) ** 2))
    else:
        # Fallback for simpler shapes or adjust as needed
        displacement_errors = np.linalg.norm(all_predictions - all_targets, axis=-1)
        ade = np.mean(displacement_errors)
        rmse = np.sqrt(np.mean((all_predictions - all_targets) ** 2))
        
    print(f'Average Displacement Error (ADE): {ade:.6f}')
    print(f'Root Mean Squared Error (RMSE): {rmse:.6f}')

    # Display predicted vs actual for the first 5 examples (first sequence, first ID, first prediction step)
    print("\nPredicted vs Actual (First 5 examples - first ID, first pred step):")
    if len(all_predictions.shape) == 4:
        print("Predicted:", all_predictions[:5, 0, 0, :])
        print("Actual:", all_targets[:5, 0, 0, :])
    else:
        print("Predicted:", all_predictions[:5])
        print("Actual:", all_targets[:5])
    return all_predictions



In [ ]:

print(csv_dir)
# feauture_scaler takes X,Y,Height,Width
df, transformer_max_ids_per_frame, frame_scaler, feature_scaler = load_and_preprocess_data(csv_folder=csv_dir)

# 2. Create tensor from dataframe
all_data_tensor = create_tensor_from_dataframe(df, transformer_max_ids_per_frame)

# 3. Create input-output sequences
X, Y = create_sequences(all_data_tensor)

model = torch.load(model_path)

/home/jaskin/Deep-Learning-Project/Data/one_csv

All CSVs now have 297 frames after trimming
Minimum records per ID: 1
Average records per ID: 202.94
Maximum records per ID: 289

Minimum IDs (Vehicles) per frame: 9
Average IDs (Vehicles) per frame: 12.30
Maximum IDs (Vehicles) per frame: 16

After normalization:
X range: 0.0000 to 5.0000
Y range: 0.0000 to 5.0000
Height range: 0.0000 to 5.0000
Width range: 0.0000 to 5.0000
Frame range: 0.0000 to 5.0000
Determined tensor ID dimension size based on max(ID_Norm): 18
All data tensor shape: torch.Size([1, 297, 18, 5])

Training on device: cuda


Training Progress:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1/50 [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 1/50 [Val]:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 2/50 [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

...

Epoch 49/50 [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 49/50 [Val]:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 50/50 [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 50/50 [Val]:   0%|          | 0/2 [00:00<?, ?it/s]


Training complete in 15.64 seconds, or 0.26 minutes
Total epochs run: 50
Average time per epoch: 0.31 seconds
Inference time per batch: 0.06 seconds
Final Training Loss: 0.7014
Final Validation Loss: 0.7194
Final Training Accuracy: 89.27%
Final Validation Accuracy: 89.55%


In [38]:
import csv
from tqdm.notebook import tqdm


def denorm_y(y, feature_scaler):
    y_flat = np.reshape(y, (y.shape[0] * y.shape[1], y.shape[2]))
    print(f"Y_flat shape: {y_flat.shape}")
    y_padded = np.pad(y_flat, ((0, 0), (0, 2)), mode='constant', constant_values=0)
    print(f"Y_padded shape: {y_padded.shape}")
    y_denorm_padded = feature_scaler.inverse_transform(y_padded)
    y_denorm_flat = y_denorm_padded[:, :2]  # Only keep x and y
    y_denorm = np.reshape(y_denorm_flat, (y.shape[0], y.shape[1], 2))
    print(f"Y_denorm shape: {y_denorm.shape}")
    print(f"Y_denorm: {y_denorm[0][0]}")
    return y_denorm
def denorm_x(x, frame_scaler, feature_scaler):
    x_flat = np.reshape(x, (x.shape[0] * x.shape[1], x.shape[2]))
    print(f"X_flat shape: {x_flat.shape}")
    frame_x = x_flat[:, 0].reshape(-1, 1)
    features_x = x_flat[:, 1:]
    print(f"feautures_x shape: {features_x.shape}")
    print(f"frame_x shape: {frame_x.shape}")
    # Normalize the features
    features_x = feature_scaler.inverse_transform(features_x)
    frame_x = frame_scaler.inverse_transform(frame_x)
    
    # Concatenate the normalized features with the frame column
    x_denorm_flat = np.concatenate((frame_x, features_x), axis=1)
    x_denorm = np.reshape(x_denorm_flat, (x.shape[0], x.shape[1], x.shape[2]))
    print(f"X_denorm shape: {x_denorm.shape}")
    print(f"X_denorm: {x_denorm[0][0]}")
    return x_denorm

def predict_130_frames(model, X, Y, csv_path, frame_scaler, feature_scaler, sequence_idx, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    headers = ['Frame', 'ID', 'X_pred', 'Y_pred', 'X_true', 'Y_true']
    print(f"Exporting predictions to {csv_path}...")
    
    with open(csv_path, 'w', newline='') as csvfile:
        csv_writer = csv.writer(csvfile)
        csv_writer.writerow(headers)
        
        with torch.no_grad():
            # X shape: (100, 18, 5)
            # Y shape: (30, 18, 2)
            
            # X contains (Seq_idx, ID, Features) 
            #    Features = [Frame, X, Y, Width, Height]
            # Y contains [Seq_idx, ID, X, Y]
            
            
            x = X[sequence_idx]
            x_unsqueezed = x.unsqueeze(0).to(device)
            y = Y[sequence_idx]
            y = y.cpu().numpy()
            y_pred = model(x_unsqueezed).squeeze(0).cpu().numpy()
            print(f"X shape: {x.shape}")
            print(f"Y shape: {y.shape}")
            print(f"Y_pred shape: {y_pred.shape}")
            
            print(f"First ID of First frame of X")
            print(x[0][0])
            print(f"First ID of First frame of Y")
            print(y[0][0])
            print(f"First ID of First frame of Y_pred")
            print(y_pred[0][0])
            
            x_denorm = denorm_x(x, frame_scaler, feature_scaler)
            y_denorm = denorm_y(y, feature_scaler)
            y_pred_denorm = denorm_y(y_pred, feature_scaler)
            
            print(f"First ID of First frame of X_denorm")
            print(x_denorm[0][0])
            print(f"First ID of First frame of Y_denorm")
            print(y_denorm[0][0])
            print(f"First ID of First frame of Y_pred_denorm")
            print(y_pred_denorm[0][0])
            
            # Write first 100 frames from x only
            # true = pred
            for frame_id, frame in enumerate(x_denorm, start=0):
                for v_id, v_id_features in enumerate(frame):
                    row = [
                        frame_id,  # Frame
                        v_id,  # ID
                        int(v_id_features[1]),  # X_pred
                        int(v_id_features[2]),  # Y_pred
                        int(v_id_features[1]),  # X_true
                        int(v_id_features[2])   # Y_true
                    ]
                    if np.any(v_id_features < 0):
                        continue
                    csv_writer.writerow(row)
            # Write next 30 frames from y and y_pred
            for seq_idx in range(30):
                y_frame = y_denorm[seq_idx]
                y_pred_frame = y_pred_denorm[seq_idx]
                for y_id, (y_id_features, y_pred_id_features) in enumerate(zip(y_frame, y_pred_frame)):
                    row = [
                        seq_idx + 100,  # Frame
                        y_id,
                        int(y_pred_id_features[0]),  # X_pred
                        int(y_pred_id_features[1]),  # Y_pred
                        int(y_id_features[0]),  # X_true
                        int(y_id_features[1]),   # Y_true
                    ]
                    if np.any(y_id_features < 0):
                        pass
                        continue
                    csv_writer.writerow(row)
    print(f"Finished exporting predictions to {csv_path}")
predict_130_frames(
    model,
    X,
    Y,
    'predictions.csv',
    frame_scaler,
    feature_scaler,
    sequence_idx=0,  # Change this to the desired sequence index
    device=None
)


Exporting predictions to predictions.csv...
X shape: torch.Size([100, 18, 5])
Y shape: (30, 18, 2)
Y_pred shape: (30, 18, 2)
First ID of First frame of X
tensor([0.0000, 4.4417, 4.0206, 2.7650, 2.9464])
First ID of First frame of Y
[2.1524289 1.5979382]
First ID of First frame of Y_pred
[2.3720088 1.6582221]
X_flat shape: torch.Size([1800, 5])
feautures_x shape: torch.Size([1800, 4])
frame_x shape: torch.Size([1800, 1])
X_denorm shape: (100, 18, 5)
X_denorm: [   0.         1710.00002394  745.99999542  167.84240913  426.32142382]
Y_flat shape: (540, 2)
Y_padded shape: (540, 4)
Y_denorm shape: (30, 18, 2)
Y_denorm: [890.00006 652.     ]
Y_flat shape: (540, 2)
Y_padded shape: (540, 4)
Y_denorm shape: (30, 18, 2)
Y_denorm: [968.65356 654.33905]
First ID of First frame of X_denorm
[   0.         1710.00002394  745.99999542  167.84240913  426.32142382]
First ID of First frame of Y_denorm
[890.00006 652.     ]
First ID of First frame of Y_pred_denorm
[968.65356 654.33905]
Finished exporting p